数据预处理：

创建一个人工数据集，并存储在 csv（逗号分隔值）文件

In [3]:
import os

# 直接在当前目录下创建 'data' 文件夹
os.makedirs('data', exist_ok=True)      # exist_ok=True，如果这个文件夹已经存在了，程序也不会报错崩溃。
# 将文件路径设置在当前目录的 'data' 文件夹内
data_file = os.path.join('data', 'house_tiny.csv')

# 写入数据到 CSV 文件
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')  # 列名
    f.write('NA,Pave,127500\n')  # 每行表示一个数据样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

# 在现实世界的数据采集中，数据缺失是常态（比如有的房子没登记房间数，或者没有巷子）。
# 在这里特意制造了一个包含缺失值的数据集，因为“如何处理这些缺失值”正是数据预处理环节最核心、最让人头疼的任务之一。

从创建的 csv 文件中加载原始数据集

In [ ]:
import pandas as pd

data = pd.read_csv(data_file)           # 读取 CSV 文件中的数据，并存储在一个 DataFrame 中
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


为了解决缺失的数据，典型的方法包括插值和删除，这里，我们将考虑插值

In [9]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]                            # 将前两列作为输入，最后一列作为输出
inputs = inputs.fillna(inputs.mean(numeric_only=True))                          # 将输入中的缺失值替换为该列的平均值
print(inputs)

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


对于 inputs 中的类别值或离散值，我们将“NAN”视为一个类别

In [10]:
inputs = pd.get_dummies(inputs, dummy_na=True)  # 将输入中的类别值转换为独热编码，并将缺失值视为一个类别
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


现在 inputs 和 outputs 中的所有条目都是数值类型，可以转换为张量格式

In [12]:
import torch

inputs = inputs.astype(float)
outputs = outputs.astype(float)

x, y = torch.tensor(inputs.values), torch.tensor(outputs.values)
x, y 

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))